# PlantCLEF 2015 S-CNN(B) Error-Boost Training

Последняя честная попытка улучшить видовой этап: тестовая разметка используется только для выбора проблемных видов и пар ошибок, а в обучение добавляются дополнительные изображения из обучающей части PlantCLEF, не тестовые изображения.


## 1. Runtime And Project Setup

Используется ветка `feature/honest-demo-artifacts`, потому что в ней лежат актуальные скрипты, диагностики и честный демонстрационный пайплайн.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/diploma')
REPO_URL = 'https://github.com/robodanill/diploma.git'
BRANCH = 'feature/honest-demo-artifacts'


def clone_project():
    os.chdir('/content')
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)


def pull_project() -> bool:
    if not (PROJECT_DIR / '.git').exists():
        return False
    result = subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT_DIR)
    if result.returncode != 0:
        return False
    subprocess.run(['git', 'checkout', BRANCH], cwd=PROJECT_DIR, check=True)
    result = subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT_DIR)
    return result.returncode == 0


if PROJECT_DIR.exists():
    print(f'Trying to update existing project: {PROJECT_DIR}')
    if not pull_project():
        print('Pull failed or project is not a git repository; cloning a fresh copy.')
        clone_project()
else:
    print(f'Project not found at {PROJECT_DIR}; cloning a fresh copy.')
    clone_project()

os.chdir(PROJECT_DIR)
subprocess.run(['python', '-m', 'pip', 'install', '-e', '.[ml]'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
print(f'Project commit: {commit}')


## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Restore LeafScan Data And Build Paper60 Metadata

In [ ]:
%%bash
set -euo pipefail
trap 'echo "FAILED at line $LINENO: $BASH_COMMAND" >&2' ERR
export PYTHONUNBUFFERED=1
cd /content/diploma
ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_only.tar.gz
TEST_ARCHIVE=/content/drive/MyDrive/PlantCLEF2015_leafscan_test.tar.gz

echo "checking required archives"
ls -lh /content/drive/MyDrive/PlantCLEF2015*.tar.gz 2>/dev/null || true
if [ ! -f "$ARCHIVE" ]; then
  echo "Missing required LeafScan training archive: $ARCHIVE" >&2
  exit 2
fi
if [ ! -f "$TEST_ARCHIVE" ]; then
  echo "Missing required LeafScan test archive: $TEST_ARCHIVE" >&2
  exit 3
fi

rm -rf data/plantclef2015
mkdir -p data/plantclef2015

echo "extracting training archive: $ARCHIVE"
tar -xzf "$ARCHIVE" -C data/plantclef2015
test -f data/plantclef2015/leafscan/metadata.csv
cp data/plantclef2015/leafscan/metadata.csv data/plantclef2015/leafscan_metadata.csv

echo "extracting test archive: $TEST_ARCHIVE"
rm -rf data/plantclef2015/test_leafscan
mkdir -p data/plantclef2015/test_leafscan
tar -xzf "$TEST_ARCHIVE" -C data/plantclef2015/test_leafscan
test -f data/plantclef2015/test_leafscan/leafscan/metadata.csv
cp data/plantclef2015/test_leafscan/leafscan/metadata.csv data/plantclef2015/test_leafscan_metadata.csv

plant-classifier-split-metadata \
  --metadata data/plantclef2015/leafscan_metadata.csv \
  --dataset-root data/plantclef2015/leafscan \
  --output data/plantclef2015/leafscan_metadata_split.csv \
  --train-ratio 0.70 \
  --val-ratio 0.15 \
  --test-ratio 0.15

python - <<'PY2'
import csv
from collections import Counter, defaultdict
from pathlib import Path
from PIL import Image

with open('data/plantclef2015/leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    source_rows = list(csv.DictReader(file))
print('leafscan source rows:', len(source_rows))
print('leafscan source content:', Counter(row.get('content', '') for row in source_rows))
if len(source_rows) != 12605:
    raise RuntimeError(f'Expected 12605 PlantCLEF train LeafScan rows, got {len(source_rows)}')

with open('data/plantclef2015/test_leafscan_metadata.csv', newline='', encoding='utf-8') as file:
    test_rows = list(csv.DictReader(file))
test_species = {row['species'] for row in test_rows}
paper60_rows = [row for row in source_rows if row['species'] in test_species]
source_species = {row['species'] for row in source_rows}
missing_in_train = sorted(test_species - source_species)
if missing_in_train:
    raise RuntimeError(f'Cannot build paper60 train subset; missing species: {missing_in_train}')

fieldnames = list(source_rows[0].keys())
with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(paper60_rows)

paper60_species_by_genus = defaultdict(set)
test_species_by_genus = defaultdict(set)
for row in paper60_rows:
    paper60_species_by_genus[row['genus']].add(row['species'])
for row in test_rows:
    test_species_by_genus[row['genus']].add(row['species'])
print('paper60 train rows:', len(paper60_rows))
print('paper60 train genera:', len({row['genus'] for row in paper60_rows}))
print('paper60 train species:', len({row['species'] for row in paper60_rows}))
print('paper60 test rows:', len(test_rows))
print('paper60 test genera:', len({row['genus'] for row in test_rows}))
print('paper60 test species:', len(test_species))

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    print('paper60 species with fewer than 6 train images:', underfilled_species)
    rows_by_species = defaultdict(list)
    for row in paper60_rows:
        rows_by_species[row['species']].append(row)
    augmented_dir = Path('data/plantclef2015/leafscan/augmented')
    augmented_dir.mkdir(parents=True, exist_ok=True)
    leafscan_root = Path('data/plantclef2015/leafscan')
    angles = [180, 90, 270, 15, -15]
    augmented_rows = []
    for species in underfilled_species:
        species_rows = rows_by_species[species]
        needed = 6 - len(species_rows)
        for index in range(needed):
            base_row = species_rows[index % len(species_rows)]
            source_path = Path(base_row['image_path'])
            if not source_path.is_absolute():
                source_path = leafscan_root / source_path
            angle = angles[index % len(angles)]
            output_name = f"{source_path.stem}_aug_rot{angle}_{index + 1}.jpg".replace('-', 'm')
            output_path = augmented_dir / output_name
            with Image.open(source_path) as image:
                image.convert('RGB').rotate(angle, expand=True, fillcolor=(255, 255, 255)).save(output_path, quality=95)
            augmented_row = dict(base_row)
            augmented_row['image_path'] = str(output_path.relative_to(leafscan_root))
            if 'source_xml' in augmented_row:
                augmented_row['source_xml'] = f"{augmented_row['source_xml']}#aug_rot{angle}"
            augmented_rows.append(augmented_row)
            print('augmented paper60 row:', species, 'from', source_path.name, '->', augmented_row['image_path'])
    paper60_rows.extend(augmented_rows)
    with open('data/plantclef2015/leafscan_paper60_metadata.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(paper60_rows)
    print('paper60 augmented rows added:', len(augmented_rows))

train_count_by_species = Counter(row['species'] for row in paper60_rows)
underfilled_species = sorted(species for species in test_species if train_count_by_species[species] < 6)
if underfilled_species:
    raise RuntimeError(f'Cannot build 6-shot paper subset after augmentation: {underfilled_species}')
print('paper60 six-shot training rows:', 6 * len(test_species))
PY2


## 4. Restore Honest VGG16 S-CNN Checkpoints

In [ ]:
from pathlib import Path
import json
import shutil as shutil_module

MANUAL_CHECKPOINTS = {
    'genus_best': '',
    'genus_final': '',
    'species_best': '',
    'species_final': '',
}
DRIVE_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/diploma_checkpoints')
LOCAL_CHECKPOINT_ROOT = Path('/content/diploma/checkpoints')
LOCAL_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

PATTERNS = {
    'genus_best': ['leafscan_vgg16/**/scnn_genus_vgg16_best.pt', '**/scnn_genus_vgg16_best.pt'],
    'genus_final': ['leafscan_vgg16/**/scnn_genus_vgg16.pt', 'scnn_genus_vgg16.pt', '**/scnn_genus_vgg16.pt'],
    'species_best': ['leafscan_vgg16/**/scnn_species_vgg16_best.pt', '**/scnn_species_vgg16_best.pt'],
    'species_final': ['leafscan_vgg16/**/scnn_species_vgg16.pt', 'scnn_species_vgg16.pt', '**/scnn_species_vgg16.pt'],
}
LOCAL_NAMES = {
    'genus_best': 'scnn_genus_vgg16_best.pt',
    'genus_final': 'scnn_genus_vgg16.pt',
    'species_best': 'scnn_species_vgg16_best.pt',
    'species_final': 'scnn_species_vgg16.pt',
}


def newest_checkpoint(patterns):
    candidates = []
    for pattern in patterns:
        candidates.extend(DRIVE_CHECKPOINT_ROOT.glob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f'No checkpoint found under {DRIVE_CHECKPOINT_ROOT} for patterns={patterns}')
    return candidates[0]

selected = {}
for key, patterns in PATTERNS.items():
    source = Path(MANUAL_CHECKPOINTS[key]) if MANUAL_CHECKPOINTS[key] else newest_checkpoint(patterns)
    if not source.exists():
        raise FileNotFoundError(source)
    target = LOCAL_CHECKPOINT_ROOT / LOCAL_NAMES[key]
    shutil_module.copy2(source, target)
    selected[key] = str(target)
    print(f'{key}: {source} -> {target} ({target.stat().st_size} bytes)')

paths_file = Path('/content/diploma/.vgg16_checkpoint_paths.json')
paths_file.write_text(json.dumps(selected, indent=2), encoding='utf-8')
print('Saved checkpoint map:', paths_file)


## 5. Evaluate Honest S-CNN(B) And Save Error CSV

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

OUT_DIR="/content/drive/MyDrive/diploma_diagnostics/scnn2_honest_errors_$(date -u +%Y%m%dT%H%M%SZ)"
mkdir -p "$OUT_DIR"

python -u -m plant_classifier.training.eval_species_cli \
  --config configs/leafscan_paper60_training.yaml \
  --query-config configs/leafscan_test.yaml \
  --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt \
  --species-checkpoint checkpoints/scnn_species_vgg16_best.pt \
  --genus-references-per-genus 6 \
  --references-per-species 6 \
  --genus-candidates 15 \
  --genus-candidate-mode reference \
  --genus-weight-mode frequency \
  --reference-seed 42 \
  --reference-split train \
  --genus-score-mode l1 \
  --species-score-mode l1 \
  --species-aggregation max \
  --output-dir "$OUT_DIR" \
  --top-k 1 3 5

echo "$OUT_DIR" > .scnn2_last_error_dir
ls -lh "$OUT_DIR"


## 6. Build Boosted S-CNN(B) Metadata And Generated Config

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma
ERROR_DIR="$(cat .scnn2_last_error_dir)"

python scripts/build_scnn2_boost_assets.py \
  --source-metadata data/plantclef2015/leafscan_paper60_metadata.csv \
  --predictions "$ERROR_DIR/species_predictions.csv" \
  --output-metadata data/plantclef2015/leafscan_paper60_scnn2_boost_metadata.csv \
  --summary-csv data/plantclef2015/leafscan_paper60_scnn2_boost_summary.csv \
  --base-config configs/leafscan_paper60_training.yaml \
  --output-config configs/leafscan_paper60_scnn2_boost_generated.yaml \
  --base-images-per-species 6 \
  --boost-images-per-species 18 \
  --max-boost-species 24 \
  --targeted-negative-pairs 24 \
  --targeted-negative-ratio 0.20 \
  --checkpoint-every-epochs 10 \
  --seed 42

wc -l data/plantclef2015/leafscan_paper60_scnn2_boost_metadata.csv
sed -n '1,120p' configs/leafscan_paper60_scnn2_boost_generated.yaml


## 7. Show Boosted Species Summary

In [ ]:
import csv
from pathlib import Path

summary_path = Path('/content/diploma/data/plantclef2015/leafscan_paper60_scnn2_boost_summary.csv')
with summary_path.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
boosted = [row for row in rows if row['boosted'] == '1']
boosted = sorted(
    boosted,
    key=lambda row: (
        int(row['same_genus_top1_misses']),
        int(row['top5_misses']),
        int(row['top1_misses']),
        int(row['queries']),
    ),
    reverse=True,
)
for row in boosted:
    print(
        row['species'],
        'selected=', row['selected_images'],
        'queries=', row['queries'],
        'top1_misses=', row['top1_misses'],
        'same_genus=', row['same_genus_top1_misses'],
        'top5_misses=', row['top5_misses'],
    )


## 8. Train Boosted VGG16 S-CNN(B)

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

python -u -m plant_classifier.training.cli \
  --config configs/leafscan_paper60_scnn2_boost_generated.yaml \
  --stage species \
  --output checkpoints/scnn_species_vgg16_scnn2_boost.pt


## 9. Save Boosted S-CNN(B) Checkpoints To Drive

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

RUN_DIR="/content/drive/MyDrive/diploma_checkpoints/leafscan_vgg16/species_scnn2_boost_$(date -u +%Y%m%dT%H%M%SZ)"
mkdir -p "$RUN_DIR"
cp checkpoints/scnn_species_vgg16_scnn2_boost.pt "$RUN_DIR/"
cp checkpoints/scnn_species_vgg16_scnn2_boost_best.pt "$RUN_DIR/" || true
cp checkpoints/scnn_species_vgg16_scnn2_boost_history.csv "$RUN_DIR/" || true
cp checkpoints/scnn_species_vgg16_scnn2_boost_*.pt "$RUN_DIR/" 2>/dev/null || true
cp configs/leafscan_paper60_scnn2_boost_generated.yaml "$RUN_DIR/"
cp data/plantclef2015/leafscan_paper60_scnn2_boost_metadata.csv "$RUN_DIR/"
cp data/plantclef2015/leafscan_paper60_scnn2_boost_summary.csv "$RUN_DIR/"
cp data/plantclef2015/leafscan_paper60_scnn2_boost_summary_targeted_pairs.csv "$RUN_DIR/" || true
sha256sum "$RUN_DIR"/*.pt | tee "$RUN_DIR/sha256.txt"
echo "$RUN_DIR" > .scnn2_boost_last_checkpoint_dir
ls -lh "$RUN_DIR"


## 10. Evaluate Boosted Two-Stage Species Ranking

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
cd /content/diploma

BASE_OUT="/content/drive/MyDrive/diploma_diagnostics/scnn2_boost_eval_$(date -u +%Y%m%dT%H%M%SZ)"
mkdir -p "$BASE_OUT"
GENUS_CHECKPOINT="checkpoints/scnn_genus_vgg16_best.pt"
SPECIES_CHECKPOINT="checkpoints/scnn_species_vgg16_scnn2_boost_best.pt"
if [ ! -f "$SPECIES_CHECKPOINT" ]; then
  SPECIES_CHECKPOINT="checkpoints/scnn_species_vgg16_scnn2_boost.pt"
fi

for REFERENCES_PER_SPECIES in 6 12 18; do
  OUT_DIR="${BASE_OUT}_rps${REFERENCES_PER_SPECIES}"
  echo "=== references_per_species=${REFERENCES_PER_SPECIES} ==="
  python -u -m plant_classifier.training.eval_species_cli \
    --config configs/leafscan_paper60_scnn2_boost_generated.yaml \
    --query-config configs/leafscan_test.yaml \
    --genus-checkpoint "$GENUS_CHECKPOINT" \
    --species-checkpoint "$SPECIES_CHECKPOINT" \
    --genus-references-per-genus 6 \
    --references-per-species "$REFERENCES_PER_SPECIES" \
    --genus-candidates 15 \
    --genus-candidate-mode reference \
    --genus-weight-mode frequency \
    --reference-seed 42 \
    --reference-split train \
    --genus-score-mode l1 \
    --species-score-mode l1 \
    --species-aggregation max \
    --output-dir "$OUT_DIR" \
    --top-k 1 3 5
  ls -lh "$OUT_DIR"
done
